# Solución implementando LightGBM

In [1]:
# EJECUTAR LIBRERIAS PARA LIGHTGBM
import os
import pandas as pd
import numpy as np

# Librerías para LightGBM
import lightgbm as lgb

# Librerías para preprocesamiento y evaluación
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Librerías para visualización (opcional)
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de warnings
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías para LightGBM cargadas correctamente")
print(f"📦 Versión de LightGBM: {lgb.__version__}")
print(f"📦 Versión de pandas: {pd.__version__}")
print(f"📦 Versión de numpy: {np.__version__}")

✅ Librerías para LightGBM cargadas correctamente
📦 Versión de LightGBM: 4.5.0
📦 Versión de pandas: 2.2.2
📦 Versión de numpy: 2.0.2


In [2]:
# LEEMOS LOS DATOS
drive_base_path = 'C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Data/'
filename = 'sell-in.txt'
filepath = os.path.join(drive_base_path, filename)
sell = pd.read_csv(filepath, sep='\t')
print(sell.head(10))

filename = 'product_id_apredecir201912.txt'
filepath = os.path.join(drive_base_path, filename)
a_predecir = pd.read_csv(filepath, sep='\t')

   periodo  customer_id  product_id  plan_precios_cuidados  cust_request_qty  \
0   201701        10234       20524                      0                 2   
1   201701        10032       20524                      0                 1   
2   201701        10217       20524                      0                 1   
3   201701        10125       20524                      0                 1   
4   201701        10012       20524                      0                11   
5   201701        10080       20524                      0                 1   
6   201701        10015       20524                      0                 4   
7   201701        10062       20524                      0                 1   
8   201701        10159       20524                      0                 3   
9   201701        10183       20524                      0                 1   

   cust_request_tn       tn  
0          0.05300  0.05300  
1          0.13628  0.13628  
2          0.03028  0.03028  

In [3]:
# Veo la cantidad de valores distintos en "periodo"
num_periodos = sell['periodo'].nunique()
print(f'Cantidad de valores distintos en "periodo": {num_periodos}')

Cantidad de valores distintos en "periodo": 36


In [4]:
# Agrupar y sumar
sell_agrup = (
    sell
    .groupby(['periodo', 'product_id'], as_index=False)['tn']
    .sum()
)

print(sell_agrup)

       periodo  product_id          tn
0       201701       20001   934.77222
1       201701       20002   550.15707
2       201701       20003  1063.45835
3       201701       20004   555.91614
4       201701       20005   494.27011
...        ...         ...         ...
31238   201912       21265     0.05007
31239   201912       21266     0.05121
31240   201912       21267     0.01569
31241   201912       21271     0.00298
31242   201912       21276     0.00892

[31243 rows x 3 columns]


In [5]:
# FEATURE ENGINEERING
# Ordenar por product_id y periodo para asegurar el orden correcto
sell_agrup = sell_agrup.sort_values(['product_id', 'periodo']).reset_index(drop=True)

# Crear las 35 columnas con los valores de los períodos anteriores (lags)
print("Creando lags (valores de períodos anteriores)...")
for i in range(1, 36):  # Del 1 al 35
    sell_agrup[f'tn_lag_{i}'] = sell_agrup.groupby('product_id')['tn'].shift(i)
    if i % 10 == 0:  # Mostrar progreso cada 10 lags
        print(f"  Creados lags hasta lag_{i}")

# Crear las columnas de delta lags (diferencias entre períodos consecutivos)
print("\nCreando delta lags (diferencias entre períodos consecutivos)...")
for i in range(1, 35):  # Del 1 al 34 (35-1)
    sell_agrup[f'tn_delta_lag_{i}'] = sell_agrup[f'tn_lag_{i}'] - sell_agrup[f'tn_lag_{i+1}']
    if i % 10 == 0:  # Mostrar progreso cada 10 delta lags
        print(f"  Creados delta lags hasta delta_lag_{i}")

# Mostrar el resultado
print("\n" + "="*60)
print("RESUMEN DEL FEATURE ENGINEERING:")
print("="*60)
print(f"✅ Creadas 35 columnas de lags (tn_lag_1 a tn_lag_35)")
print(f"✅ Creadas 34 columnas de delta lags (tn_delta_lag_1 a tn_delta_lag_34)")
print(f"📊 Total de nuevas features: 69 columnas")

print(f"\n📋 Forma del dataset: {sell_agrup.shape}")
print(f"📋 Total de columnas: {len(sell_agrup.columns)}")

print(f"\n🔍 Primeras filas del dataset:")
print(sell_agrup.head(10))

print(f"\n📝 Lista completa de columnas:")
print(f"Originales: {list(sell_agrup.columns[:3])}")
print(f"Lags: tn_lag_1 hasta tn_lag_35 ({len([col for col in sell_agrup.columns if 'tn_lag_' in col and 'delta' not in col])} columnas)")
print(f"Delta lags: tn_delta_lag_1 hasta tn_delta_lag_34 ({len([col for col in sell_agrup.columns if 'tn_delta_lag_' in col])} columnas)")

Creando lags (valores de períodos anteriores)...
  Creados lags hasta lag_10
  Creados lags hasta lag_20
  Creados lags hasta lag_30

Creando delta lags (diferencias entre períodos consecutivos)...
  Creados delta lags hasta delta_lag_10
  Creados delta lags hasta delta_lag_20
  Creados delta lags hasta delta_lag_30

RESUMEN DEL FEATURE ENGINEERING:
✅ Creadas 35 columnas de lags (tn_lag_1 a tn_lag_35)
✅ Creadas 34 columnas de delta lags (tn_delta_lag_1 a tn_delta_lag_34)
📊 Total de nuevas features: 69 columnas

📋 Forma del dataset: (31243, 72)
📋 Total de columnas: 72

🔍 Primeras filas del dataset:
   periodo  product_id          tn    tn_lag_1    tn_lag_2    tn_lag_3  \
0   201701       20001   934.77222         NaN         NaN         NaN   
1   201702       20001   798.01620   934.77222         NaN         NaN   
2   201703       20001  1303.35771   798.01620   934.77222         NaN   
3   201704       20001  1069.96130  1303.35771   798.01620   934.77222   
4   201705       20001  1

In [6]:
# AGREGAR FEATURES TEMPORALES BASADAS EN EL CAMPO PERIODO
import calendar
from datetime import datetime

print("=== CREANDO FEATURES TEMPORALES ===")
print("Extrayendo información temporal del campo 'periodo'...")

# Extraer año y mes del campo periodo (formato: YYYYMM)
sell_agrup['año'] = sell_agrup['periodo'] // 100  # Primeros 4 dígitos
sell_agrup['mes'] = sell_agrup['periodo'] % 100   # Últimos 2 dígitos

# Crear función para obtener días del mes
def obtener_dias_mes(año, mes):
    """Obtiene la cantidad de días de un mes específico de un año dado"""
    try:
        return calendar.monthrange(año, mes)[1]
    except:
        return 30  # Valor por defecto en caso de error

# Aplicar función para obtener días del mes
sell_agrup['dias_mes'] = sell_agrup.apply(
    lambda row: obtener_dias_mes(int(row['año']), int(row['mes'])), 
    axis=1
)

# Crear función para obtener quarter (trimestre)
def obtener_quarter(mes):
    """Obtiene el quarter del año basado en el mes"""
    if mes in [1, 2, 3]:
        return 1
    elif mes in [4, 5, 6]:
        return 2
    elif mes in [7, 8, 9]:
        return 3
    elif mes in [10, 11, 12]:
        return 4
    else:
        return 1  # Valor por defecto

# Aplicar función para obtener quarter
sell_agrup['quarter'] = sell_agrup['mes'].apply(obtener_quarter)

# Mostrar resumen de las nuevas columnas creadas
print(f"\n✅ FEATURES TEMPORALES CREADAS:")
print(f"   - año: Año extraído de periodo")
print(f"   - mes: Mes extraído de periodo") 
print(f"   - dias_mes: Cantidad de días del mes")
print(f"   - quarter: Trimestre del año (1-4)")

print(f"\n📊 RESUMEN DE DATOS TEMPORALES:")
print(f"Años únicos: {sorted(sell_agrup['año'].unique())}")
print(f"Meses únicos: {sorted(sell_agrup['mes'].unique())}")
print(f"Quarters únicos: {sorted(sell_agrup['quarter'].unique())}")

print(f"\n🔍 EJEMPLO DE DATOS TEMPORALES:")
print("periodo | año | mes | dias_mes | quarter")
print("-" * 40)
sample_data = sell_agrup[['periodo', 'año', 'mes', 'dias_mes', 'quarter']].head(10)
for _, row in sample_data.iterrows():
    print(f"{row['periodo']:>7} | {row['año']:>4} | {row['mes']:>3} | {row['dias_mes']:>8} | {row['quarter']:>7}")

print(f"\n📈 ESTADÍSTICAS DE DÍAS POR MES:")
dias_stats = sell_agrup.groupby('mes')['dias_mes'].first().sort_index()
for mes, dias in dias_stats.items():
    nombre_mes = calendar.month_name[mes]
    print(f"  Mes {mes:2d} ({nombre_mes:<9}): {dias} días")

print(f"\n📋 NUEVA FORMA DEL DATASET:")
print(f"Shape: {sell_agrup.shape}")
print(f"Nuevas columnas agregadas: {['año', 'mes', 'dias_mes', 'quarter']}")
print(f"Total de columnas temporales: 4")

# Verificar si hay valores nulos en las nuevas columnas
print(f"\n🔍 VERIFICACIÓN DE CALIDAD:")
for col in ['año', 'mes', 'dias_mes', 'quarter']:
    nulos = sell_agrup[col].isnull().sum()
    if nulos > 0:
        print(f"⚠️  {col}: {nulos} valores nulos")
    else:
        print(f"✅ {col}: Sin valores nulos")

print(f"\n📊 DISTRIBUCIÓN POR QUARTER:")
quarter_dist = sell_agrup['quarter'].value_counts().sort_index()
for quarter, count in quarter_dist.items():
    print(f"  Quarter {quarter}: {count:,} registros")
    
print(f"\n🎯 Features temporales listas para usar en LightGBM!")

=== CREANDO FEATURES TEMPORALES ===
Extrayendo información temporal del campo 'periodo'...

✅ FEATURES TEMPORALES CREADAS:
   - año: Año extraído de periodo
   - mes: Mes extraído de periodo
   - dias_mes: Cantidad de días del mes
   - quarter: Trimestre del año (1-4)

📊 RESUMEN DE DATOS TEMPORALES:
Años únicos: [np.int64(2017), np.int64(2018), np.int64(2019)]
Meses únicos: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
Quarters únicos: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

🔍 EJEMPLO DE DATOS TEMPORALES:
periodo | año | mes | dias_mes | quarter
----------------------------------------
 201701 | 2017 |   1 |       31 |       1
 201702 | 2017 |   2 |       28 |       1
 201703 | 2017 |   3 |       31 |       1
 201704 | 2017 |   4 |       30 |       2
 201705 | 2017 |   5 |       31 |       2
 201706 | 2017 |   6 |       30 |       2
 201707 | 2017 |   7 |     

In [7]:
# Crear el subconjunto filtrado por los product_id a predecir
sell_agrup_subset = sell_agrup[sell_agrup['product_id'].isin(a_predecir['product_id'])]

print(f"Dataset original shape: {sell_agrup.shape}")
print(f"Subconjunto filtrado shape: {sell_agrup_subset.shape}")
print(f"Cantidad de productos únicos en el subconjunto: {sell_agrup_subset['product_id'].nunique()}")
print("\nPrimeras filas del subconjunto:")
print(sell_agrup_subset.head())

Dataset original shape: (31243, 76)
Subconjunto filtrado shape: (22349, 76)
Cantidad de productos únicos en el subconjunto: 780

Primeras filas del subconjunto:
   periodo  product_id          tn    tn_lag_1    tn_lag_2   tn_lag_3  \
0   201701       20001   934.77222         NaN         NaN        NaN   
1   201702       20001   798.01620   934.77222         NaN        NaN   
2   201703       20001  1303.35771   798.01620   934.77222        NaN   
3   201704       20001  1069.96130  1303.35771   798.01620  934.77222   
4   201705       20001  1502.20132  1069.96130  1303.35771  798.01620   

    tn_lag_4  tn_lag_5  tn_lag_6  tn_lag_7  ...  tn_delta_lag_29  \
0        NaN       NaN       NaN       NaN  ...              NaN   
1        NaN       NaN       NaN       NaN  ...              NaN   
2        NaN       NaN       NaN       NaN  ...              NaN   
3        NaN       NaN       NaN       NaN  ...              NaN   
4  934.77222       NaN       NaN       NaN  ...             

In [8]:
# IMPORTAR PREDICCIONES DE AUTOGLUON Y AGREGAR COMO FEATURE
print("=== IMPORTANDO PREDICCIONES DE AUTOGLUON ===")

# Importar el archivo de predicciones AutoGluon
autogluon_path = 'C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/AutoGluon/predicciones_febrero2020_nvw2.csv'
auto_glo = pd.read_csv(autogluon_path)

print(f"📂 Archivo AutoGluon cargado exitosamente")
print(f"📊 Shape de auto_glo: {auto_glo.shape}")
print(f"📋 Columnas de auto_glo: {list(auto_glo.columns)}")

print(f"\n🔍 Primeras filas de auto_glo:")
print(auto_glo.head())

print(f"\n📈 Estadísticas de la columna 'tn' de AutoGluon:")
if 'tn' in auto_glo.columns:
    print(f"  Cantidad de registros: {len(auto_glo)}")
    print(f"  Valores únicos: {auto_glo['tn'].nunique()}")
    print(f"  Min: {auto_glo['tn'].min():.2f}")
    print(f"  Max: {auto_glo['tn'].max():.2f}")
    print(f"  Media: {auto_glo['tn'].mean():.2f}")
    print(f"  Valores nulos: {auto_glo['tn'].isnull().sum()}")
else:
    print("⚠️  La columna 'tn' no se encontró en auto_glo")
    print(f"Columnas disponibles: {list(auto_glo.columns)}")

# Verificar si hay columna product_id para el merge
if 'product_id' in auto_glo.columns:
    print(f"\n🔗 Preparando merge con sell_agrup_subset...")
    print(f"📊 Products en auto_glo: {auto_glo['product_id'].nunique()}")
    print(f"📊 Products en sell_agrup_subset: {sell_agrup_subset['product_id'].nunique()}")
    
    # Verificar productos en común
    productos_comunes = set(auto_glo['product_id']).intersection(set(sell_agrup_subset['product_id']))
    print(f"📊 Productos en común: {len(productos_comunes)}")
    
    if len(productos_comunes) > 0:
        # Renombrar la columna tn de AutoGluon para evitar conflictos
        auto_glo_feature = auto_glo[['product_id', 'tn']].copy()
        auto_glo_feature = auto_glo_feature.rename(columns={'tn': 'autogluon_pred'})
        
        # Realizar el merge con sell_agrup_subset
        print(f"\n🔄 Realizando merge...")
        sell_agrup_subset_original_shape = sell_agrup_subset.shape
        
        sell_agrup_subset = sell_agrup_subset.merge(
            auto_glo_feature, 
            on='product_id', 
            how='left'
        )
        
        print(f"✅ Merge completado!")
        print(f"📊 Shape antes del merge: {sell_agrup_subset_original_shape}")
        print(f"📊 Shape después del merge: {sell_agrup_subset.shape}")
        
        # Verificar la nueva columna
        if 'autogluon_pred' in sell_agrup_subset.columns:
            print(f"\n📈 Estadísticas de la nueva feature 'autogluon_pred':")
            print(f"  Valores no nulos: {sell_agrup_subset['autogluon_pred'].notna().sum()}")
            print(f"  Valores nulos: {sell_agrup_subset['autogluon_pred'].isnull().sum()}")
            print(f"  Min: {sell_agrup_subset['autogluon_pred'].min():.2f}")
            print(f"  Max: {sell_agrup_subset['autogluon_pred'].max():.2f}")
            print(f"  Media: {sell_agrup_subset['autogluon_pred'].mean():.2f}")
            
            print(f"\n🔍 Muestra de datos con la nueva feature:")
            print(sell_agrup_subset[['product_id', 'periodo', 'tn', 'autogluon_pred']].head(10))
            
            # Verificar si hay valores nulos y mostrar estrategia
            nulos_autogluon = sell_agrup_subset['autogluon_pred'].isnull().sum()
            if nulos_autogluon > 0:
                print(f"\n⚠️  ATENCIÓN: {nulos_autogluon} productos no tienen predicción AutoGluon")
                print("💡 Estrategias posibles:")
                print("   1. Imputar con la media de AutoGluon")
                print("   2. Imputar con 0")
                print("   3. Excluir productos sin predicción AutoGluon")
                
                # Aplicar estrategia por defecto: imputar con la media
                media_autogluon = sell_agrup_subset['autogluon_pred'].mean()
                sell_agrup_subset['autogluon_pred'].fillna(media_autogluon, inplace=True)
                print(f"✅ Valores nulos imputados con la media: {media_autogluon:.2f}")
            
            print(f"\n🎯 Feature AutoGluon agregada exitosamente!")
            print(f"📋 Nueva lista de columnas (últimas 5): {list(sell_agrup_subset.columns)[-5:]}")
            
        else:
            print("❌ Error: No se pudo agregar la feature AutoGluon")
    else:
        print("❌ Error: No hay productos en común entre los datasets")
else:
    print("❌ Error: No se encontró la columna 'product_id' en auto_glo para realizar el merge")

print(f"\n📊 RESUMEN FINAL:")
print(f"Dataset auto_glo: {auto_glo.shape}")
print(f"Dataset sell_agrup_subset: {sell_agrup_subset.shape}")
print(f"Nueva feature agregada: {'autogluon_pred' if 'autogluon_pred' in sell_agrup_subset.columns else 'Ninguna'}")
print(f"Total features temporales + lags + delta lags + AutoGluon: {sell_agrup_subset.shape[1] - 3} features")  # -3 por periodo, product_id, tn

=== IMPORTANDO PREDICCIONES DE AUTOGLUON ===
📂 Archivo AutoGluon cargado exitosamente
📊 Shape de auto_glo: (780, 2)
📋 Columnas de auto_glo: ['product_id', 'tn']

🔍 Primeras filas de auto_glo:
   product_id           tn
0       20001  1310.471755
1       20002  1068.112072
2       20003   714.680381
3       20004   533.487580
4       20005   521.103435

📈 Estadísticas de la columna 'tn' de AutoGluon:
  Cantidad de registros: 780
  Valores únicos: 780
  Min: 0.02
  Max: 1310.47
  Media: 36.60
  Valores nulos: 0

🔗 Preparando merge con sell_agrup_subset...
📊 Products en auto_glo: 780
📊 Products en sell_agrup_subset: 780
📊 Productos en común: 780

🔄 Realizando merge...
✅ Merge completado!
📊 Shape antes del merge: (22349, 76)
📊 Shape después del merge: (22349, 77)

📈 Estadísticas de la nueva feature 'autogluon_pred':
  Valores no nulos: 22349
  Valores nulos: 0
  Min: 0.02
  Max: 1310.47
  Media: 42.16

🔍 Muestra de datos con la nueva feature:
   product_id  periodo          tn  autogluon_p

In [9]:
# SPLIT TEMPORAL PARA TRAIN/VALIDATION - PREDICCIÓN 202002
print("=== SPLIT TEMPORAL PARA SERIES DE TIEMPO ===")

# Analizar los períodos disponibles en el dataset
periodos_disponibles = sorted(sell_agrup_subset['periodo'].unique())
print(f"📅 Períodos disponibles: {periodos_disponibles}")
print(f"📊 Total de períodos: {len(periodos_disponibles)}")

# ESTRATEGIA DE SPLIT TEMPORAL MODIFICADA:
# - OBJETIVO: Predecir 202002 (febrero 2020)
# - TRAIN: Períodos hasta 201910 (octubre 2019) 
# - VALIDATION: 201911 Y 201912 (noviembre y diciembre 2019) - más datos para validación
# - TEST: 202001 (enero 2020) - opcional para evaluación final

print(f"\n🎯 ESTRATEGIA DE SPLIT TEMPORAL MODIFICADA:")
print(f"   • OBJETIVO: Predecir 202002 (febrero 2020)")
print(f"   • TRAIN: Hasta 201910 (octubre 2019) - para entrenamiento")
print(f"   • VALIDATION: 201911 Y 201912 (nov-dic 2019) - simula predicción 2-3 meses adelante")
print(f"   • FUTURE: 202001 (enero 2020) - opcional para evaluación final")

# Definir los cortes temporales
periodo_corte_train = 201910  # Último período para train
periodos_validation = [201911, 201912]  # Períodos para validation
periodo_objetivo = 202002     # Período que queremos predecir

# Realizar el split temporal
train_data = sell_agrup_subset[sell_agrup_subset['periodo'] <= periodo_corte_train].copy()
val_data = sell_agrup_subset[sell_agrup_subset['periodo'].isin(periodos_validation)].copy()

print(f"\n📊 RESULTADOS DEL SPLIT:")
print(f"   • Dataset original: {sell_agrup_subset.shape}")
print(f"   • TRAIN (≤ {periodo_corte_train}): {train_data.shape}")
print(f"   • VALIDATION ({periodos_validation}): {val_data.shape}")

# Verificar distribución temporal
print(f"\n📈 DISTRIBUCIÓN TEMPORAL TRAIN:")
train_periodos = train_data['periodo'].value_counts().sort_index()
for periodo, count in train_periodos.items():
    print(f"   {periodo}: {count:,} registros")

print(f"\n📈 DISTRIBUCIÓN TEMPORAL VALIDATION:")
val_periodos = val_data['periodo'].value_counts().sort_index()
for periodo, count in val_periodos.items():
    print(f"   {periodo}: {count:,} registros")

# Verificar que tenemos los mismos productos en train y val
productos_train = set(train_data['product_id'].unique())
productos_val = set(val_data['product_id'].unique())
productos_comunes = productos_train.intersection(productos_val)

print(f"\n🔍 VERIFICACIÓN DE PRODUCTOS:")
print(f"   • Productos únicos en TRAIN: {len(productos_train)}")
print(f"   • Productos únicos en VALIDATION: {len(productos_val)}")
print(f"   • Productos en común: {len(productos_comunes)}")
print(f"   • Productos solo en TRAIN: {len(productos_train - productos_val)}")
print(f"   • Productos solo en VALIDATION: {len(productos_val - productos_train)}")

# Preparar features y target para el entrenamiento
# Identificar columnas de features (excluir periodo, product_id, tn)
feature_columns = [col for col in train_data.columns 
                  if col not in ['periodo', 'product_id', 'tn']]

print(f"\n🎯 PREPARACIÓN DE FEATURES:")
print(f"   • Total de features disponibles: {len(feature_columns)}")
print(f"   • Features incluyen:")
print(f"     - Lags (35): tn_lag_1 a tn_lag_35")
print(f"     - Delta lags (34): tn_delta_lag_1 a tn_delta_lag_34") 
print(f"     - Temporales (4): año, mes, dias_mes, quarter")
print(f"     - AutoGluon (1): autogluon_pred")

# Filtrar productos que tengan datos tanto en train como en validation
productos_validos = list(productos_comunes)
train_data_filtered = train_data[train_data['product_id'].isin(productos_validos)].copy()
val_data_filtered = val_data[val_data['product_id'].isin(productos_validos)].copy()

print(f"\n📊 DATASETS FILTRADOS (solo productos comunes):")
print(f"   • TRAIN filtrado: {train_data_filtered.shape}")
print(f"   • VALIDATION filtrado: {val_data_filtered.shape}")

# Eliminar filas con NaN en las features (especialmente en lags)
print(f"\n🧹 LIMPIEZA DE DATOS:")
print(f"   • NaN en TRAIN antes de limpieza: {train_data_filtered[feature_columns].isnull().sum().sum()}")
print(f"   • NaN en VALIDATION antes de limpieza: {val_data_filtered[feature_columns].isnull().sum().sum()}")

# Eliminar filas con NaN
train_clean = train_data_filtered.copy()  # Se hace la prueba SIN eliminar los NaN
val_clean = val_data_filtered.copy()      # Se hace la prueba SIN eliminar los NaN

print(f"   • TRAIN después de limpieza: {train_clean.shape}")
print(f"   • VALIDATION después de limpieza: {val_clean.shape}")

# Preparar X y y para train y validation
X_train = train_clean[feature_columns]
y_train = train_clean['tn']

X_val = val_clean[feature_columns]
y_val = val_clean['tn']

print(f"\n✅ DATASETS FINALES PREPARADOS:")
print(f"   • X_train: {X_train.shape}")
print(f"   • y_train: {y_train.shape}")
print(f"   • X_val: {X_val.shape}")
print(f"   • y_val: {y_val.shape}")

print(f"\n📋 FEATURES UTILIZADAS ({len(feature_columns)}):")
for i, feature in enumerate(feature_columns, 1):
    if i <= 10:  # Mostrar primeras 10
        print(f"   {i:2d}. {feature}")
    elif i == 11:
        print(f"   ... ({len(feature_columns)-10} features adicionales)")

print(f"\n🎯 DATOS LISTOS PARA ENTRENAMIENTO DE LIGHTGBM!")
print(f"💡 PRÓXIMOS PASOS:")
print(f"   1. Entrenar LightGBM con X_train, y_train (hasta 201910)")
print(f"   2. Validar con X_val, y_val (201911 + 201912 - simula predicción 3-4 meses adelante)")
print(f"   3. Optimizar hiperparámetros usando esta división temporal")
print(f"   4. Aplicar modelo final para predecir 202002")
print(f"   5. Más datos de validación = mejor estimación del performance real")

=== SPLIT TEMPORAL PARA SERIES DE TIEMPO ===
📅 Períodos disponibles: [np.int64(201701), np.int64(201702), np.int64(201703), np.int64(201704), np.int64(201705), np.int64(201706), np.int64(201707), np.int64(201708), np.int64(201709), np.int64(201710), np.int64(201711), np.int64(201712), np.int64(201801), np.int64(201802), np.int64(201803), np.int64(201804), np.int64(201805), np.int64(201806), np.int64(201807), np.int64(201808), np.int64(201809), np.int64(201810), np.int64(201811), np.int64(201812), np.int64(201901), np.int64(201902), np.int64(201903), np.int64(201904), np.int64(201905), np.int64(201906), np.int64(201907), np.int64(201908), np.int64(201909), np.int64(201910), np.int64(201911), np.int64(201912)]
📊 Total de períodos: 36

🎯 ESTRATEGIA DE SPLIT TEMPORAL MODIFICADA:
   • OBJETIVO: Predecir 202002 (febrero 2020)
   • TRAIN: Hasta 201910 (octubre 2019) - para entrenamiento
   • VALIDATION: 201911 Y 201912 (nov-dic 2019) - simula predicción 2-3 meses adelante
   • FUTURE: 202001 

In [10]:
# ENTRENAMIENTO LIGHTGBM - VALIDACIÓN Y MODELO FINAL
print("=== ENTRENAMIENTO LIGHTGBM PARA PREDICCIÓN 202002 ===")

# Verificar que tenemos los datos preparados
print(f"\n📊 VERIFICACIÓN DE DATOS PREPARADOS:")
print(f"   • X_train: {X_train.shape}")
print(f"   • y_train: {y_train.shape}")
print(f"   • X_val: {X_val.shape}")  
print(f"   • y_val: {y_val.shape}")

# Verificar si tenemos datos suficientes para entrenar
if X_train.shape[0] == 0:
    print(f"\n❌ ERROR CRÍTICO: No hay datos de entrenamiento disponibles!")
    print(f"💡 DIAGNÓSTICO:")
    print(f"   • X_train vacío: {X_train.shape}")
    print(f"   • X_val disponible: {X_val.shape}")
    print(f"   • Causa probable: Filtrado muy agresivo elimina todos los datos de train")
    
    print(f"\n🔧 SOLUCIÓN: Usar estrategia de split alternativa")
    print(f"   • Combinar datos disponibles y hacer split 80/20")
    print(f"   • Mantener respeto temporal en la medida de lo posible")
    
    # Combinar datos de val que tenemos disponibles
    print(f"\n🔄 IMPLEMENTANDO SPLIT ALTERNATIVO...")
    
    # Usar datos de validación disponibles y hacer split temporal
    datos_disponibles = pd.concat([val_clean], ignore_index=True)
    print(f"   • Datos totales disponibles: {datos_disponibles.shape}")
    
    # Ordenar por período para mantener cierto respeto temporal
    datos_disponibles = datos_disponibles.sort_values(['product_id', 'periodo']).reset_index(drop=True)
    
    # Split 80/20 respetando productos
    productos_unicos = datos_disponibles['product_id'].unique()
    n_train = int(len(datos_disponibles) * 0.8)
    
    # Tomar primeros registros para train, últimos para val
    train_alt = datos_disponibles.iloc[:n_train].copy()
    val_alt = datos_disponibles.iloc[n_train:].copy()
    
    # Actualizar X_train, y_train, X_val, y_val
    X_train = train_alt[feature_columns]
    y_train = train_alt['tn']
    X_val = val_alt[feature_columns]
    y_val = val_alt['tn']
    
    print(f"   ✅ Split alternativo aplicado:")
    print(f"   • X_train: {X_train.shape}")
    print(f"   • y_train: {y_train.shape}")
    print(f"   • X_val: {X_val.shape}")
    print(f"   • y_val: {y_val.shape}")

# Verificación final antes de entrenar
if X_train.shape[0] == 0 or X_val.shape[0] == 0:
    print(f"\n❌ ABORTANDO: Aún no hay datos suficientes para entrenar")
    raise ValueError("Cannot train without sufficient data in both train and validation sets")

print(f"\n✅ DATOS VERIFICADOS - PROCEDIENDO CON ENTRENAMIENTO")

# ==========================================
# PASO 1: ENTRENAMIENTO INICIAL CON VALIDACIÓN
# ==========================================
print(f"\n🎯 PASO 1: ENTRENAMIENTO INICIAL CON VALIDACIÓN")
print(f"Training hasta 201910, Validación en 201911-201912")

# Configurar parámetros LightGBM optimizados para series temporales
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'linear_tree': True,  # Activa regresión lineal en cada hoja
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'random_state': 42,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'min_child_samples': 20,
    'min_data_in_leaf': 10,
    'max_depth': 6
}

print(f"\n📋 PARÁMETROS LIGHTGBM:")
for param, value in lgb_params.items():
    print(f"   • {param}: {value}")

# Crear datasets de LightGBM
print(f"\n🔄 Creando datasets LightGBM...")
train_data_lgb = lgb.Dataset(X_train, label=y_train)
val_data_lgb = lgb.Dataset(X_val, label=y_val, reference=train_data_lgb)

# Entrenar modelo inicial con validación
print(f"\n🚀 ENTRENANDO MODELO INICIAL...")
callbacks = [
    lgb.early_stopping(stopping_rounds=50),
    lgb.log_evaluation(period=100)
]

model_initial = lgb.train(
    lgb_params,
    train_data_lgb,
    valid_sets=[train_data_lgb, val_data_lgb],
    valid_names=['train', 'val'],
    num_boost_round=1000,
    callbacks=callbacks
)

print(f"\n✅ MODELO INICIAL ENTRENADO!")
print(f"   • Mejor iteración: {model_initial.best_iteration}")
print(f"   • Mejor score validación: {model_initial.best_score['val']['rmse']:.4f}")

# Realizar predicciones en validación
print(f"\n📈 EVALUANDO EN VALIDACIÓN...")
y_val_pred = model_initial.predict(X_val, num_iteration=model_initial.best_iteration)

# Calcular métricas de validación
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))
mae_val = mean_absolute_error(y_val, y_val_pred)
r2_val = r2_score(y_val, y_val_pred)

print(f"\n📊 MÉTRICAS DE VALIDACIÓN:")
print(f"   • RMSE: {rmse_val:.4f}")
print(f"   • MAE: {mae_val:.4f}")
print(f"   • R²: {r2_val:.4f}")

# Mostrar importancia de features (top 15)
print(f"\n🎯 TOP 15 FEATURES MÁS IMPORTANTES:")
feature_importance = model_initial.feature_importance(importance_type='gain')
feature_names = X_train.columns
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

for i, row in importance_df.head(15).iterrows():
    print(f"   {row.name+1:2d}. {row['feature']:<20}: {row['importance']:>8.0f}")

# ==========================================
# PASO 2: MODELO FINAL CON DATOS EXTENDIDOS
# ==========================================
print(f"\n" + "="*60)
print(f"🎯 PASO 2: ENTRENAMIENTO MODELO FINAL")
print(f"Incluyendo 201911 y 201912 en entrenamiento para predecir 202002")
print(f"="*60)

# Preparar datos extendidos (train + validation)
print(f"\n🔄 PREPARANDO DATOS EXTENDIDOS...")

# Si usamos split alternativo, necesitamos datos adicionales para el modelo final
# Intentar usar todos los datos temporales disponibles hasta 201912
print(f"\n🔍 BUSCANDO DATOS ADICIONALES HASTA 201912...")

# Obtener todos los datos hasta 201912 para el modelo final
datos_hasta_201912 = sell_agrup_subset[sell_agrup_subset['periodo'] <= 201912].copy()
print(f"   • Datos hasta 201912: {datos_hasta_201912.shape}")

# Limpiar datos finales
datos_final_clean = datos_hasta_201912.dropna(subset=feature_columns).copy()
print(f"   • Datos finales limpiados: {datos_final_clean.shape}")

if datos_final_clean.shape[0] > 0:
    # Usar todos los datos hasta 201912 para el modelo final
    X_final = datos_final_clean[feature_columns]
    y_final = datos_final_clean['tn']
    print(f"   • Usando datos históricos completos hasta 201912")
else:
    # Fallback: combinar train y validation actual
    X_final = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
    y_final = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)
    print(f"   • Usando combinación train + val actual")

print(f"   • X_final: {X_final.shape}")
print(f"   • y_final: {y_final.shape}")

# Crear dataset final para LightGBM
print(f"\n🔄 CREANDO DATASET FINAL...")

# Verificar que tenemos datos suficientes
if X_final.shape[0] == 0:
    print(f"❌ ERROR: No hay datos para el modelo final")
    raise ValueError("No data available for final model training")

train_final_lgb = lgb.Dataset(X_final, label=y_final)
print(f"   ✅ Dataset final creado: {X_final.shape}")

# Entrenar modelo final con más iteraciones (sin early stopping)
print(f"\n🚀 ENTRENANDO MODELO FINAL...")

# Ajustar parámetros para modelo final
lgb_params_final = lgb_params.copy()
lgb_params_final['learning_rate'] = 0.03  # Learning rate más bajo para más estabilidad
lgb_params_final['num_leaves'] = 25       # Menos hojas para evitar overfitting

# Usar iteraciones basadas en el modelo inicial o valor por defecto
try:
    num_iterations_final = int(model_initial.best_iteration * 1.2)  # 20% más iteraciones
except:
    num_iterations_final = 500  # Valor por defecto si no hay best_iteration
    print(f"   ⚠️  Usando iteraciones por defecto: {num_iterations_final}")

model_final = lgb.train(
    lgb_params_final,
    train_final_lgb,
    num_boost_round=num_iterations_final,
    callbacks=[lgb.log_evaluation(period=200)]
)

print(f"\n✅ MODELO FINAL ENTRENADO!")
print(f"   • Iteraciones utilizadas: {model_final.num_trees()}")

# ==========================================
# PASO 3: PREPARAR DATOS PARA PREDICCIÓN 202002
# ==========================================
print(f"\n" + "="*60)
print(f"🎯 PASO 3: PREDICCIÓN PARA 202002")
print(f"="*60)

# Necesitamos crear datos para 202002 usando los últimos valores disponibles
print(f"\n🔄 PREPARANDO DATOS PARA PREDICCIÓN 202002...")

# Tomar el último período disponible por producto para crear el registro 202002
ultimo_periodo_por_producto = sell_agrup_subset.groupby('product_id')['periodo'].max()
print(f"   • Último período por producto (muestra): {ultimo_periodo_por_producto.head()}")

# Filtrar productos únicos para la predicción
productos_a_predecir = a_predecir['product_id'].tolist()
print(f"   • Productos a predecir: {len(productos_a_predecir)}")

# Crear datos para 202002 basados en el último registro de cada producto
print(f"\n📋 CREANDO REGISTROS PARA 202002...")
registros_202002 = []

for product_id in productos_a_predecir:
    # Obtener el último registro del producto
    ultimo_registro = sell_agrup_subset[
        sell_agrup_subset['product_id'] == product_id
    ].sort_values('periodo').tail(1)
    
    if len(ultimo_registro) > 0:
        nuevo_registro = ultimo_registro.copy()
        
        # Actualizar el período a 202002
        nuevo_registro['periodo'] = 202002
        nuevo_registro['año'] = 2020
        nuevo_registro['mes'] = 2
        nuevo_registro['dias_mes'] = 29  # 2020 es año bisiesto
        nuevo_registro['quarter'] = 1
        
        # Actualizar lags: el valor actual se convierte en lag_1, etc.
        valor_actual = ultimo_registro['tn'].iloc[0]
        
        # Shift de todos los lags
        for i in range(35, 1, -1):  # De lag_35 a lag_2
            if f'tn_lag_{i-1}' in nuevo_registro.columns:
                nuevo_registro[f'tn_lag_{i}'] = ultimo_registro[f'tn_lag_{i-1}'].iloc[0]
        
        # El valor actual se convierte en lag_1
        nuevo_registro['tn_lag_1'] = valor_actual
        
        # Actualizar delta lags
        for i in range(1, 35):
            if f'tn_lag_{i}' in nuevo_registro.columns and f'tn_lag_{i+1}' in nuevo_registro.columns:
                nuevo_registro[f'tn_delta_lag_{i}'] = (
                    nuevo_registro[f'tn_lag_{i}'].iloc[0] - nuevo_registro[f'tn_lag_{i+1}'].iloc[0]
                )
        
        # Mantener la predicción AutoGluon (es para 202002)
        # No modificar autogluon_pred
        
        # Remover la columna tn (es lo que queremos predecir)
        nuevo_registro = nuevo_registro.drop(columns=['tn'])
        
        registros_202002.append(nuevo_registro)

# Combinar todos los registros
if registros_202002:
    datos_202002 = pd.concat(registros_202002, ignore_index=True)
    print(f"   ✅ Registros 202002 creados: {datos_202002.shape}")
    
    # Preparar features para predicción
    X_pred_202002 = datos_202002[feature_columns]
    print(f"   • X_pred_202002: {X_pred_202002.shape}")
    
    # Verificar que no hay valores nulos
    nulos_pred = X_pred_202002.isnull().sum().sum()
    print(f"   • Valores nulos en features: {nulos_pred}")
    
    if nulos_pred > 0:
        print(f"   ⚠️  Imputando valores nulos...")
        # Usar forward fill y luego llenar con 0
        try:
            X_pred_202002 = X_pred_202002.fillna(method='ffill').fillna(0)
        except:
            # Para versiones más nuevas de pandas
            X_pred_202002 = X_pred_202002.ffill().fillna(0)
        print(f"   ✅ Valores nulos imputados")
    
    # ==========================================
    # PASO 4: REALIZAR PREDICCIÓN FINAL
    # ==========================================
    print(f"\n🚀 REALIZANDO PREDICCIÓN PARA 202002...")
    
    # Hacer predicción
    predicciones_202002 = model_final.predict(X_pred_202002)
    
    # Crear DataFrame con resultados
    resultados_202002 = pd.DataFrame({
        'product_id': datos_202002['product_id'],
        'tn_predicted': predicciones_202002
    })
    
    print(f"\n✅ PREDICCIONES COMPLETADAS!")
    print(f"   • Productos predichos: {len(resultados_202002)}")
    print(f"   • Período objetivo: 202002")
    
    # Estadísticas de las predicciones
    print(f"\n📊 ESTADÍSTICAS DE PREDICCIONES:")
    print(f"   • Min: {predicciones_202002.min():.4f}")
    print(f"   • Max: {predicciones_202002.max():.4f}")
    print(f"   • Media: {predicciones_202002.mean():.4f}")
    print(f"   • Mediana: {np.median(predicciones_202002):.4f}")
    print(f"   • Std: {predicciones_202002.std():.4f}")
    
    # Mostrar primeras predicciones
    print(f"\n🔍 PRIMERAS 10 PREDICCIONES:")
    print(resultados_202002.head(10))
    
    # Guardar resultados
    output_path = 'C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/LightGBM'
    os.makedirs(output_path, exist_ok=True)
    
    # Guardar predicciones
    resultados_path = os.path.join(output_path, 'predicciones_202002_lightgbm.csv')
    resultados_202002.to_csv(resultados_path, index=False)
    print(f"\n💾 RESULTADOS GUARDADOS:")
    print(f"   📁 {resultados_path}")
    
    # Guardar modelo final
    model_path = os.path.join(output_path, 'lightgbm_model_final.txt')
    model_final.save_model(model_path)
    print(f"   📁 {model_path}")
    
    # Comparar con AutoGluon si disponible
    if 'autogluon_pred' in datos_202002.columns:
        print(f"\n📈 COMPARACIÓN CON AUTOGLUON:")
        autogluon_preds = datos_202002['autogluon_pred'].values
        correlacion = np.corrcoef(predicciones_202002, autogluon_preds)[0, 1]
        print(f"   • Correlación LightGBM vs AutoGluon: {correlacion:.4f}")
        print(f"   • Media AutoGluon: {autogluon_preds.mean():.4f}")
        print(f"   • Media LightGBM: {predicciones_202002.mean():.4f}")
        
        # Agregar columna de comparación
        resultados_202002['autogluon_pred'] = autogluon_preds
        resultados_202002['diferencia'] = resultados_202002['tn_predicted'] - resultados_202002['autogluon_pred']
        
        print(f"\n🔍 DIFERENCIAS (LightGBM - AutoGluon):")
        print(f"   • Diferencia promedio: {resultados_202002['diferencia'].mean():.4f}")
        print(f"   • Diferencia std: {resultados_202002['diferencia'].std():.4f}")
    
    print(f"\n🎉 PROCESO COMPLETADO EXITOSAMENTE!")
    print(f"   ✅ Modelo entrenado y validado")
    print(f"   ✅ Modelo final con datos extendidos")
    print(f"   ✅ Predicciones para 202002 generadas")
    print(f"   ✅ Resultados guardados en formato CSV")
    
else:
    print(f"❌ Error: No se pudieron crear registros para predicción 202002")

print(f"\n" + "="*60)
print(f"🏁 ENTRENAMIENTO LIGHTGBM FINALIZADO")
print(f"="*60)

=== ENTRENAMIENTO LIGHTGBM PARA PREDICCIÓN 202002 ===

📊 VERIFICACIÓN DE DATOS PREPARADOS:
   • X_train: (20789, 74)
   • y_train: (20789,)
   • X_val: (1560, 74)
   • y_val: (1560,)

✅ DATOS VERIFICADOS - PROCEDIENDO CON ENTRENAMIENTO

🎯 PASO 1: ENTRENAMIENTO INICIAL CON VALIDACIÓN
Training hasta 201910, Validación en 201911-201912

📋 PARÁMETROS LIGHTGBM:
   • objective: regression
   • metric: rmse
   • boosting_type: gbdt
   • linear_tree: True
   • num_leaves: 31
   • learning_rate: 0.05
   • feature_fraction: 0.9
   • bagging_fraction: 0.8
   • bagging_freq: 5
   • verbose: -1
   • random_state: 42
   • reg_alpha: 0.1
   • reg_lambda: 0.1
   • min_child_samples: 20
   • min_data_in_leaf: 10
   • max_depth: 6

🔄 Creando datasets LightGBM...

🚀 ENTRENANDO MODELO INICIAL...
Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[46]	train's rmse: 32.005	val's rmse: 26.9534

✅ MODELO 